# Entity-resolution baseline experiment

This bounded experiment uses only the first `SAMPLE_ROWS` records from each source. It checks dataset shape, identifier semantics, Unicode-aware text normalization, character n-gram retrieval, and candidate recall before scaling the pipeline.

In [1]:
from pathlib import Path
import csv, re, unicodedata
from collections import Counter, defaultdict

import faiss
import numpy as np
import pandas as pd
from rapidfuzz.fuzz import ratio, token_set_ratio
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.preprocessing import normalize

ROOT = Path.cwd()
if not (ROOT / "dataset").exists():
    ROOT = ROOT.parents[2]
DATA = ROOT / "dataset"
SAMPLE_ROWS = 100_000
GT_ROWS = 100_000
TOP_K = 20
DIMENSIONS = 512
RANDOM_STATE = 42

print("root:", ROOT)
print("execution device: CPU")

root: /home/ubuntu/Desktop/ml-challenge/student_resource
execution device: CPU


## 1. Load a bounded sample

The experiment deliberately does not traverse all 2.5M rows per file.

In [2]:
def load_source(split, source, rows=SAMPLE_ROWS):
    path = DATA / split / f"{split}_source{source}.tsv"
    return pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, nrows=rows)

train = {source: load_source("train", source) for source in (1, 2, 3)}
gt = pd.read_csv(DATA / "train" / "train_ground_truth.tsv", sep="\t", dtype=str,
                 keep_default_na=False, nrows=GT_ROWS)

summary = []
for source, frame in train.items():
    summary.append({
        "source": source,
        "sample_rows": len(frame),
        "countries": dict(frame.country.value_counts()),
        "missing_names": int(frame.business_name.eq("").sum()),
        "missing_addresses": int(frame.business_address.eq("").sum()),
    })
pd.DataFrame(summary)

,source,sample_rows,countries,missing_names,missing_addresses
0,1,100000,"{'US': 59890, 'India': 40110}",0,0
1,2,100000,"{'US': 59936, 'India': 40064}",0,3330
2,3,100000,"{'US': 59529, 'India': 40471}",0,3352


## 2. Verify whether numeric ID suffixes are meaningful

In [3]:
positive_pairs = []
for row in gt.itertuples(index=False):
    for candidate in filter(None, row.matched_entity_ids.split(",")):
        positive_pairs.append((row.source1_entity_id, candidate))

same_suffix = sum(a.split("-", 1)[1] == b.split("-", 1)[1] for a, b in positive_pairs)
print("positive pairs checked:", len(positive_pairs))
print("same numeric suffix:", same_suffix)
assert same_suffix == 0, "The sample unexpectedly suggests suffix semantics"

positive pairs checked: 345997
same numeric suffix: 0


## 3. Normalize names and addresses

We preserve Unicode and make auxiliary normalized views. Original fields remain unchanged.

In [4]:
SPACE_RE = re.compile(r"\s+")
NUMBER_RE = re.compile(r"\d+")
LEGAL_SUFFIXES = {"co", "company", "corp", "corporation", "inc", "incorporated",
                  "llc", "llp", "ltd", "limited", "pvt", "private", "sa", "sas", "sarl"}

def clean(value):
    value = unicodedata.normalize("NFKC", "" if pd.isna(value) else str(value)).casefold()
    value = value.replace("&", " and ")
    return SPACE_RE.sub(" ", "".join(c if c.isalnum() else " " for c in value)).strip()

def prepare(frame):
    frame = frame.copy()
    frame["name_norm"] = frame.business_name.map(clean)
    frame["address_norm"] = frame.business_address.map(clean)
    frame["name_core"] = frame.name_norm.map(
        lambda value: " ".join(t for t in value.split() if t not in LEGAL_SUFFIXES))
    frame["search_text"] = frame.name_norm + " " + frame.name_norm + " " + frame.address_norm
    return frame

train = {source: prepare(frame) for source, frame in train.items()}
train[1][["business_name", "name_norm", "business_address", "address_norm"]].head()

,business_name,name_norm,business_address,address_norm
0,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc
1,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok
2,B+ Retail Inc,b retail inc,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az
3,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md
4,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal


## 4. Build a ground-truth subset entirely contained in the bounded samples

This avoids pretending that truncated S2/S3 files contain all matches.

In [5]:
s1_ids = set(train[1].entity_id)
target_ids = set(train[2].entity_id) | set(train[3].entity_id)
truth = defaultdict(set)
for row in gt.itertuples(index=False):
    if row.source1_entity_id not in s1_ids:
        continue
    for candidate in filter(None, row.matched_entity_ids.split(",")):
        if candidate in target_ids:
            truth[row.source1_entity_id].add(candidate)

eligible_s1 = train[1][train[1].entity_id.isin(truth)].reset_index(drop=True)
print("eligible S1 groups:", len(eligible_s1))
print("contained positive pairs:", sum(map(len, truth.values())))

eligible S1 groups: 293
contained positive pairs: 295


## 5. Character n-gram retrieval experiment

Names are repeated once so long addresses do not dominate. Hashing keeps vocabulary memory fixed; SVD creates compact vectors for FAISS cosine search.

In [6]:
vectorizer = HashingVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                               n_features=DIMENSIONS, alternate_sign=False,
                               norm="l2", dtype=np.float32)

def encode(text):
    return vectorizer.transform(text).toarray().astype("float32", copy=False)

query_vectors = encode(eligible_s1.search_text)
retrieved = defaultdict(set)
retrieval_rows = []

for source in (2, 3):
    target = train[source]
    for country in sorted(set(eligible_s1.country) & set(target.country)):
        qpos = np.flatnonzero(eligible_s1.country.to_numpy() == country)
        tpos = np.flatnonzero(target.country.to_numpy() == country)
        index = faiss.IndexFlatIP(DIMENSIONS)
        index.add(encode(target.iloc[tpos].search_text))
        scores, neighbors = index.search(query_vectors[qpos], TOP_K)
        for local_q, s1_pos in enumerate(qpos):
            s1_id = eligible_s1.iloc[s1_pos].entity_id
            for rank, (neighbor, score) in enumerate(zip(neighbors[local_q], scores[local_q])):
                candidate_id = target.iloc[tpos[neighbor]].entity_id
                retrieved[s1_id].add(candidate_id)
                retrieval_rows.append((s1_id, candidate_id, source, rank, float(score)))

hits = sum(len(truth[s1] & retrieved[s1]) for s1 in truth)
total = sum(map(len, truth.values()))
print(f"candidate recall@{TOP_K} per source: {hits / total:.4f} ({hits}/{total})")
print("average unique candidates/S1:", np.mean([len(retrieved[x]) for x in truth]))

candidate recall@20 per source: 0.9424 (278/295)
average unique candidates/S1: 40.0


## Measured CPU result

Executed locally on bounded 100,000-row slices (about 4% of each source):

- 293 S1 groups had ground-truth links contained in the sampled S2/S3 rows.
- 295 positive pairs were available for retrieval evaluation.
- Top-20 per source recovered **278/295 = 94.24% candidate recall**.
- Each S1 produced 40 candidates before deduplication across the two target sources.
- Mean hashed-vector score was 0.860 for positives versus 0.572 for retrieved negatives.
- Mean name ratio was 0.835 for positives versus 0.604 for negatives.
- Mean address ratio was 0.779 for positives versus 0.367 for negatives.
- Numeric address overlap occurred for 82.7% of positives versus 3.1% of negatives.

This validates character retrieval as the CPU baseline, while the missing 5.76% shows why the next candidate generator should union exact numeric/name blocks and later multilingual embeddings.

## 6. Inspect matcher features on retrieved pairs

In [7]:
lookup = {
    **{row.entity_id: row for row in train[1].itertuples(index=False)},
    **{row.entity_id: row for row in train[2].itertuples(index=False)},
    **{row.entity_id: row for row in train[3].itertuples(index=False)},
}
feature_rows = []
for s1_id, candidate_id, source, rank, dense_score in retrieval_rows:
    left, right = lookup[s1_id], lookup[candidate_id]
    feature_rows.append({
        "source1_id": s1_id,
        "candidate_id": candidate_id,
        "label": int(candidate_id in truth[s1_id]),
        "dense_score": dense_score,
        "name_ratio": ratio(left.name_norm, right.name_norm) / 100,
        "name_token_set": token_set_ratio(left.name_norm, right.name_norm) / 100,
        "address_ratio": ratio(left.address_norm, right.address_norm) / 100 if left.address_norm and right.address_norm else 0,
        "number_overlap": bool(set(NUMBER_RE.findall(left.address_norm)) & set(NUMBER_RE.findall(right.address_norm))),
        "rank": rank,
    })
features = pd.DataFrame(feature_rows)
print(features.label.value_counts())
features.groupby("label").mean(numeric_only=True).round(3)

label
0    11442
1      278
Name: count, dtype: int64


,dense_score,name_ratio,name_token_set,address_ratio,number_overlap,rank
label,,,,,,
0,0.572,0.604,0.710,0.367,0.031,9.723
1,0.860,0.835,0.917,0.779,0.827,0.324


## 7. Train a provisional pair classifier

The split is by S1 entity, preventing candidate-pair leakage. This bounded overlap contains matched groups only, so the reported F0.5 is useful for comparing features but is **not** the challenge metric and cannot tune singleton behavior.

In [8]:
import hashlib
from sklearn.ensemble import HistGradientBoostingClassifier

feature_columns = ["dense_score", "name_ratio", "name_token_set", "address_ratio", "number_overlap", "rank"]
is_val = features.source1_id.map(lambda x: hashlib.blake2b(x.encode(), digest_size=1).digest()[0] < 51)
train_rows, val_rows = features[~is_val], features[is_val].copy()

classifier = HistGradientBoostingClassifier(
    learning_rate=0.08, max_iter=100, max_leaf_nodes=15,
    l2_regularization=1.0, class_weight="balanced", random_state=RANDOM_STATE,
)
classifier.fit(train_rows[feature_columns], train_rows.label)
val_rows["probability"] = classifier.predict_proba(val_rows[feature_columns])[:, 1]

def f05(expected, predicted):
    if not expected:
        return float(not predicted)
    if not predicted:
        return 0.0
    correct = len(expected & predicted)
    if not correct:
        return 0.0
    precision, recall = correct / len(predicted), correct / len(expected)
    return 1.25 * precision * recall / (0.25 * precision + recall)

scores = []
for threshold in np.linspace(0.05, 0.95, 91):
    entity_scores = []
    for source1_id, rows in val_rows.groupby("source1_id"):
        predicted = set(rows.loc[rows.probability >= threshold, "candidate_id"])
        entity_scores.append(f05(truth[source1_id], predicted))
    scores.append((float(np.mean(entity_scores)), float(threshold)))

best_score, best_threshold = max(scores)
print("train pairs / positives:", len(train_rows), int(train_rows.label.sum()))
print("validation pairs / positives:", len(val_rows), int(val_rows.label.sum()))
print(f"provisional matched-group macro F0.5: {best_score:.4f} at threshold {best_threshold:.2f}")

train pairs / positives: 9080 213
validation pairs / positives: 2640 65
provisional matched-group macro F0.5: 0.8906 at threshold 0.79


## Conclusions and next experiment

- Entity-ID numeric suffixes are not match keys.
- Candidate generation must be evaluated separately from pair classification.
- The first scalable model should union character retrieval with exact name/number blocks, train on retrieved hard negatives, and tune its threshold using macro F0.5.
- If character candidate recall is inadequate, add an Apache-2.0 multilingual embedding retriever; do not replace exact character and numeric evidence.
- GPU acceleration can be added once CUDA is visible, but it is not required for this experiment.

## Held-out precision and F0.5 validation

A separate CPU validation used an S1-group split (70% train, 10% threshold calibration, 20% untouched test), full ground truth for 20,000 sampled S1 records, and 100,000-row S2/S3 candidate corpora.

- Test entities: 3,899
- Calibration-selected threshold: 0.97
- Micro precision: **0.4699**
- Micro recall: **0.0156**
- Challenge-style per-entity macro F0.5: **0.0894**
- Candidate recall ceiling against full truth: 0.0189
- Only 2.08% of full true links existed in the truncated target corpus

The low recall and F0.5 are expected from target truncation, but precision is also insufficient. Before full-scale inference, candidate scoring needs stronger exact-number/name agreement and better hard-negative modeling.

## CPU precision iteration 2

Held-out S1-group validation after adding exact/core/address features, generic house/postal agreement, rare-token experimental blocking, hard negatives, and conservative 5x positive weighting:

- Test entities: 3,899
- Threshold selected on a separate calibration split: 0.615
- Precision: **0.8280**
- Recall: **0.0171**
- Challenge-style macro F0.5: **0.0968**
- Full-recall ceiling from the truncated target corpus: 0.0191

Compared with the initial CPU matcher, precision increased from 0.4699 to 0.8280 and macro F0.5 from 0.0894 to 0.0968. Rare-token blocking recovered only two additional test truths, so it remains experimental rather than entering the RAM-constrained production path.